# 00 · 실행 환경 및 `data/raw` 데이터 검증 (로컬 / Colab 겸용)

### [0] 환경 부트스트랩 및 원본 파일 점검
- **데이터 범위**: `공통 환경 설정`
- **작업 단계**: `데이터 로딩 및 환경 검증`
- **목적 및 원리**: 로컬 VS Code(`.venv`)와 Google Colab 양쪽에서 `StopIteration` 없이 프로젝트 루트를 자동 탐색하고, 원본 CSV 9종과 핵심 패키지를 확인한다.
- **확인할 내용**: `9/9 CSVs ready`, `pyarrow`/`scikit-learn`/`tqdm` 정상 임포트.

In [ ]:
# =====================================================================
# [데이터: 공통] [단계: 환경 부트스트랩] 로컬/Colab 겸용 루트 탐색
# =====================================================================
import os
import sys
from pathlib import Path

def find_project_root() -> Path:
    candidates = []
    if "NFL_PROJECT_ROOT" in os.environ:
        candidates.append(Path(os.environ["NFL_PROJECT_ROOT"]))
    candidates.extend([Path.cwd(), *Path.cwd().parents, Path("/content/nfl-bdb2027")])
    if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        candidates.extend([
            Path("/content/drive/MyDrive/nfl-bdb2027"),
            Path("/content/drive/MyDrive/Colab Notebooks/nfl-bdb2027"),
        ])
    for p in candidates:
        if (p / "src" / "config.py").exists():
            return p.resolve()
    raise FileNotFoundError(
        f"src/config.py를 찾지 못했습니다 (현재 cwd={Path.cwd()}). "
        "Colab에서는 Drive(/content/drive/MyDrive/nfl-bdb2027)에 저장소를 두거나 "
        "환경변수 NFL_PROJECT_ROOT를 지정하세요."
    )

ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config
config.setup()

In [ ]:
# =====================================================================
# [데이터: 공통] [단계: 검증] data/raw CSV 9종 존재 여부 및 용량 확인
# =====================================================================
import pandas as pd

rows = []
for key, path in config.all_raw_files().items():
    exists = path.exists()
    size_mb = round(path.stat().st_size / (1024 ** 2), 2) if exists else 0.0
    rows.append({"key": key, "filename": path.name, "exists": exists, "size_mb": size_mb})

raw_status = pd.DataFrame(rows)
display(raw_status)
assert raw_status["exists"].all(), "누락된 원본 CSV 파일이 있습니다!"
print(f"[확인된 결과] 원본 CSV {len(raw_status)}종 모두 정상 연결 (총 {raw_status['size_mb'].sum():.1f} MB)")

In [ ]:
# =====================================================================
# [데이터: 공통] [단계: 검증] Parquet(pyarrow) · scikit-learn 임포트 확인
# =====================================================================
import pyarrow
import sklearn
import tqdm

print("[확인된 결과] pyarrow:", pyarrow.__version__, "| sklearn:", sklearn.__version__, "| tqdm:", tqdm.__version__)